In [5]:
from utils import create_new_dataset_with_ipcw_weights, create_surv_data, train_test_split_into_df, get_Nbi, ipc_weighted_mse, inf_JK_bagged_variance
from lifelines import KaplanMeierFitter, WeibullAFTFitter
from sksurv.metrics import concordance_index_ipcw
from sksurv.util import Surv
from sklearn.model_selection import train_test_split
import seaborn as sns
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier

### create dataset and fit weibull modell ( with transformation of bmi and kreatininkinase)

In [3]:
n = 1000
seed = 42

data = create_surv_data(shape_weibull=1,   # constant hazard
                        scale_weibull_base=10000, 
                        rate_censoring=0.01,  
                        b_bloodp=-0.405, 
                        b_diab=-0.4, 
                        b_age=-0.05, 
                        b_bmi=-0.01, 
                        b_kreat=-0.2,
                        n=n, 
                        seed=seed)

data2 = data.copy()

data['bmi'] = (data['bmi']-25)**2
data['kreatinkinase'] = np.log(data['kreatinkinase'])

X = data[['bmi', 'blood_pressure', 'kreatinkinase', 'diabetes', 'age']]
y = Surv.from_arrays(event=data['event'], time=data['t'])

df_train, df_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=seed)
df_train, df_test = train_test_split_into_df(df_train=df_train, df_test=df_test, y_train=y_train, y_test=y_test)

tau = 12

kmf = KaplanMeierFitter()
kmf.fit(df_train['time'], event_observed=1-df_train['event'])
df_train = create_new_dataset_with_ipcw_weights(data=df_train,t=tau, kmf=kmf)
df_test = create_new_dataset_with_ipcw_weights(data=df_test,t=tau, kmf=kmf)

aft = WeibullAFTFitter()
aft.fit(df_train.drop(['weights_ipcw', 'survived'], axis=1), duration_col='time', event_col='event')
#aft.print_summary()

Data shape: (1000, 7)
34.0 % of the data has an event


<lifelines.WeibullAFTFitter: fitted with 700 total observations, 462 right-censored observations>

### Weibull Modell Evaluation

In [4]:
# IPCW-MSE berechnen
y_pred = aft.predict_survival_function(df_test.drop(['weights_ipcw', 'survived','time','event'], axis=1), times = tau).iloc[0].tolist()
weibull_mse = ipc_weighted_mse(y_true=df_test['survived'].values, y_pred=y_pred, sample_weight=df_test['weights_ipcw'])
print(f'weibull_ipcw_mse auf test data: {round(weibull_mse,4)}')


# IPCW-C-Index berechnen
tied_risk_scores = aft.predict_expectation(df_test)
ci_ipcw, concordant, discordant, tied_risk, tied_time = concordance_index_ipcw(
    y_train, y_test, -tied_risk_scores )
print("IPCW-C-Index:", round(ci_ipcw,4))

X_erwartung = pd.DataFrame({'bmi': [(25-25)**2], 'blood_pressure': [0], 'kreatinkinase': [np.log(np.exp(5+1/2))], 'diabetes': [0], 'age': [50]})
y_pred_X_erwartung_weibull = aft.predict_survival_function(X_erwartung, times = tau).iloc[0].tolist()
print('\n')
print(f'Prediction für X_erwartung zum überleben am Zeitpunkt tau={tau}: {y_pred_X_erwartung_weibull[0]:.4f}')


weibull_ipcw_mse auf test data: 0.0658
IPCW-C-Index: 0.6743


Prediction für X_erwartung zum überleben am Zeitpunkt tau=12: 0.9592
